# **Notebook 04 — EssentialAI LLM-Only Baseline**

**Module:** 2 — Expanded Multi-Model Evaluation  
**System:** `EssentialAI/rnj-1-instruct` without RAG  
**Purpose:** Re-execute the frozen Track 1 and Track 2 benchmark under the Module 2 runtime so that RAG impact can be compared against a contemporaneous standalone baseline.


# **1. Environment Initialisation**

## **1.1. Load Dependencies**

### **Install All Required Libraries**

In [1]:
# =============================================================================
# ESSENTIALAI LLM-ONLY SETUP
# INSTALL DEPENDENCIES
# =============================================================================

# Install the libraries required for telecom RAG data acquisition,
# document processing, embeddings and vector retrieval.
#
# The current runtime is CPU-based because inference is not yet being
# performed. GPU-specific acceleration can be enabled later when required.

!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    safetensors \
    sentencepiece \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    python-pptx \
    beautifulsoup4 \
    pyarrow \
    tqdm

print("All required Module 2 libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 20.2 MB/s eta 0:00:00
All required Module 2 libraries installed successfully.


### **Import All Required Libraries**

In [2]:
# =============================================================================
# ESSENTIALAI LLM-ONLY SETUP
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# ---------------------------------------------------------------------------
# Core scientific stack
# ---------------------------------------------------------------------------
import numpy as np
import scipy
import pandas as pd

# ---------------------------------------------------------------------------
# Standard Python libraries
# ---------------------------------------------------------------------------
import os
import gc
import json
import shutil
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Progress monitoring
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm

# Data / document processing
# ---------------------------------------------------------------------------
import pyarrow
import pyarrow.parquet as pq
from docx import Document
from bs4 import BeautifulSoup
from pypdf import PdfReader

# ---------------------------------------------------------------------------
# HTTP / source acquisition
# ---------------------------------------------------------------------------
import requests

# ---------------------------------------------------------------------------
# PyTorch
# ---------------------------------------------------------------------------
import torch

# ---------------------------------------------------------------------------
# Hugging Face / LLM inference
# ---------------------------------------------------------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from huggingface_hub import login, HfApi, snapshot_download

# ---------------------------------------------------------------------------
# Embeddings
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Vector similarity search
# ---------------------------------------------------------------------------
import faiss

# ---------------------------------------------------------------------------
# PDF document processing
# ---------------------------------------------------------------------------
from pypdf import PdfReader

print("All required libraries imported successfully.")

# Display key package versions for reproducibility
print("\nPackage Versions")
print("-" * 40)
print(f"NumPy           : {np.__version__}")
print(f"SciPy           : {scipy.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"FAISS           : {faiss.__version__}")
print(f"PyArrow         : {pyarrow.__version__}")

All required libraries imported successfully.

Package Versions
----------------------------------------
NumPy           : 2.1.3
SciPy           : 1.16.3
Pandas          : 2.2.3
PyTorch         : 2.11.0+cu128
FAISS           : 1.15.0
PyArrow         : 18.1.0


### **Import Datasets from Kaggle**

In [3]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [4]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cliffordimaguezegie_telecom_benchmark_path = kagglehub.dataset_download('cliffordimaguezegie/benchmark')

print('Data source import complete.')


100%|██████████| 21.9k/21.9k [00:00<00:00, 26.9MB/s]

Extracting files...
Data source import complete.


In [5]:
print("QUESTIONS:")
print(cliffordimaguezegie_telecom_benchmark_path)

QUESTIONS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [6]:
from pathlib import Path


BENCHMARK_DIR = Path(
    cliffordimaguezegie_telecom_benchmark_path
)

print("BENCHMARK :", BENCHMARK_DIR)

BENCHMARK : /root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [7]:
# =============================================================================
# RAG V1 — INSPECT BENCHMARK DATASET
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK DATASET CONTENTS")
print("=" * 90)

for file_path in sorted(
    BENCHMARK_DIR.rglob("*")
):

    if file_path.is_file():

        print(
            file_path.relative_to(
                BENCHMARK_DIR
            )
        )

print("=" * 90)

RAG V1 — BENCHMARK DATASET CONTENTS
track1_20_questions.json
track2_final_32_questions.json


In [8]:
# =============================================================================
# LOAD BENCHMARK QUESTION BANKS
# =============================================================================

import json


TRACK1_FILE = (
    BENCHMARK_DIR
    / "track1_20_questions.json"
)

TRACK2_FILE = (
    BENCHMARK_DIR
    / "track2_final_32_questions.json"
)


# =============================================================================
# LOAD TRACK 1
# =============================================================================

with open(
    TRACK1_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions = json.load(
        file
    )


# =============================================================================
# LOAD TRACK 2
# =============================================================================

with open(
    TRACK2_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions_track2 = json.load(
        file
    )


# =============================================================================
# VALIDATION
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK QUESTION BANKS LOADED")
print("=" * 90)

print("\nTRACK 1")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions[0]['id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions[-1]['id']}"
)


print("\nTRACK 2")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions_track2)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions_track2[0]['evaluation_id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions_track2[-1]['evaluation_id']}"
)


# =============================================================================
# COUNT CHECKS
# =============================================================================

if len(benchmark_questions) != 20:

    raise RuntimeError(
        f"Track 1 expected 20 questions, "
        f"found {len(benchmark_questions)}."
    )


if len(benchmark_questions_track2) != 32:

    raise RuntimeError(
        f"Track 2 expected 32 questions, "
        f"found {len(benchmark_questions_track2)}."
    )


print("\n" + "=" * 90)
print("TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED")
print("=" * 90)

RAG V1 — BENCHMARK QUESTION BANKS LOADED

TRACK 1
------------------------------------------------------------
Questions : 20
First ID  : Q01
Last ID   : Q20

TRACK 2
------------------------------------------------------------
Questions : 32
First ID  : T2-01
Last ID   : T2-32

TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED


## **1.2. Load EssentialAI General LLM**

In [9]:
# =============================================================================
# LOAD GENERAL LLM
# =============================================================================

import time
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


# =============================================================================
# GENERAL LLM CONFIGURATION
# =============================================================================

GENERAL_LLM_NAME = (
    "EssentialAI/rnj-1-instruct"
)

GENERAL_TEMPERATURE = 0.0
GENERAL_TOP_K = 50
GENERAL_TOP_P = 0.95

GENERAL_MAX_NEW_TOKENS = 512

GENERAL_QUANT_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


# =============================================================================
# GPU STATUS
# =============================================================================

print("=" * 90)
print("GENERAL LLM")
print("=" * 90)

print(
    f"GPU             : "
    f"{torch.cuda.get_device_name(0)}"
)

print(
    f"GPU memory      : "
    f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
)


# =============================================================================
# LOAD TOKENIZER
# =============================================================================

print("\nLoading General LLM tokenizer...")

tokenizer_start = time.time()

general_tokenizer = (
    AutoTokenizer.from_pretrained(
        GENERAL_LLM_NAME,
        trust_remote_code=True,
    )
)

tokenizer_elapsed = (
    time.time()
    - tokenizer_start
)


# =============================================================================
# LOAD MODEL
# =============================================================================

print(
    "\nLoading General LLM model..."
)

model_start = time.time()

general_model = (
    AutoModelForCausalLM.from_pretrained(
        GENERAL_LLM_NAME,
        quantization_config=GENERAL_QUANT_CONFIG,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
)

general_model.eval()

model_elapsed = (
    time.time()
    - model_start
)


# =============================================================================
# VALIDATION
# =============================================================================

print("\nConfiguration")
print("-" * 60)

print(
    f"Model            : "
    f"{GENERAL_LLM_NAME}"
)

print(
    f"Quantization     : 4-bit NF4"
)

print(
    f"Compute dtype    : float16"
)

print(
    f"Temperature      : "
    f"{GENERAL_TEMPERATURE}"
)

print(
    f"Top-k            : "
    f"{GENERAL_TOP_K}"
)

print(
    f"Top-p            : "
    f"{GENERAL_TOP_P}"
)

print(
    f"Max new tokens   : "
    f"{GENERAL_MAX_NEW_TOKENS}"
)

print(
    f"Tokenizer time   : "
    f"{tokenizer_elapsed:.2f} sec"
)

print(
    f"Model load time  : "
    f"{model_elapsed:.2f} sec"
)

print(
    f"\nDevice map       : "
    f"{getattr(general_model, 'hf_device_map', 'N/A')}"
)


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("GENERAL LLM LOADED SUCCESSFULLY")
print("=" * 90)

GENERAL LLM
GPU             : NVIDIA L4
GPU memory      : 22.03 GB

Loading General LLM tokenizer...


config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'full_attention', 'sliding_attention'}


tokenizer_config.json:   0%|          | 0.00/55.9k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/418 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]


Configuration
------------------------------------------------------------
Model            : EssentialAI/rnj-1-instruct
Quantization     : 4-bit NF4
Compute dtype    : float16
Temperature      : 0.0
Top-k            : 50
Top-p            : 0.95
Max new tokens   : 512
Tokenizer time   : 4.29 sec
Model load time  : 152.92 sec

Device map       : N/A

GENERAL LLM LOADED SUCCESSFULLY


# **2. Inference Pipeline**

## **2.1. Response Function**

In [10]:
import torch

def generate_response(
    model,
    tokenizer,
    question,
    system_prompt=None,
    max_new_tokens=500,
    do_sample=False,
    temperature=0.01,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
):
    """Generate a standardised response from a loaded LLM."""

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": question})

    # Use the model's native chat template when available.
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt = question

    # Tokenise the formatted prompt.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    # Place inputs on the model's execution device.
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Define generation parameters explicitly.
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
    }

    if do_sample:
        generation_kwargs.update({
            "temperature": temperature,
            "top_p": top_p,
            "top_k": top_k,
        })

    # Generate response without gradient calculation.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    # Decode only newly generated tokens.
    input_length = inputs["input_ids"].shape[-1]

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True,
    )

    return response.strip()

## **2.2. Test Response Function**

In [11]:
print("=" * 80)
print("TEST RUN: GENERAL LLM RESPONSE")
print("=" * 80)

general_response = generate_response(
    model=general_model,
    tokenizer=general_tokenizer,
    question="What is the role of the AMF in a 5G Standalone network?",
    system_prompt="You are an expert telecommunications network engineer."
)

print(general_response)

TEST RUN: GENERAL LLM RESPONSE


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In a 5G Standalone (5G-SA) network, the Access Management Function (AMF) plays a critical role as part of the 5G Core Network. The AMF is responsible for managing and controlling access to the 5G services and resources within the core network. Here are some key functions and responsibilities of the AMF:

1. **Authentication and Authorization**: The AMF authenticates users and authorizes them to access specific 5G services based on their subscription profiles and policies.

2. **Session Management**: It manages user sessions, including session establishment, modification, and termination. This includes handling mobility management between different 5G cells or across different networks.

3. **Policy Enforcement**: The AMF enforces policies related to service level agreements (SLAs), data usage, roaming, and other service-specific rules.

4. **Network Slicing**: In a network-sliced environment, the AMF is responsible for allocating and managing network slices according to the requirement

# **3. Telecom Benchmark Evaluation**

## **EssentialAI LLM-Only Inference**

### **EssentialAI Generation Configuration**

In [12]:
# =============================================================================
# GENERAL LLM GENERATION CONFIGURATION
# =============================================================================

GENERAL_GENERATION_CONFIG = {
    "temperature": 0.01,
    "top_k": 50,
    "top_p": 0.95,
    "repetition_penalty": 1.05,
    "max_new_tokens": 1024,
    "do_sample": True,
}


print("=" * 90)
print("GENERAL LLM GENERATION CONFIGURATION")
print("=" * 90)

print(
    f"Temperature        : "
    f"{GENERAL_GENERATION_CONFIG['temperature']}"
)

print(
    f"Top-k              : "
    f"{GENERAL_GENERATION_CONFIG['top_k']}"
)

print(
    f"Top-p              : "
    f"{GENERAL_GENERATION_CONFIG['top_p']}"
)

print(
    f"Repetition penalty : "
    f"{GENERAL_GENERATION_CONFIG['repetition_penalty']}"
)

print(
    f"Max new tokens     : "
    f"{GENERAL_GENERATION_CONFIG['max_new_tokens']}"
)

print(
    f"Do sample          : "
    f"{GENERAL_GENERATION_CONFIG['do_sample']}"
)

print("=" * 90)

GENERAL LLM GENERATION CONFIGURATION
Temperature        : 0.01
Top-k              : 50
Top-p              : 0.95
Repetition penalty : 1.05
Max new tokens     : 1024
Do sample          : True


### **Track 1 Inference**

In [13]:
import gc
import json
import time
from datetime import datetime, timezone
import torch

# =============================================================================
# HELPER: AGGRESSIVE HARDWARE SYNCHRONIZED MEMORY CLEARING
# =============================================================================

def flush_vram(delay_sec: float = 0.5):
    """
    Forces garbage collection, releases cached CUDA VRAM back to the GPU,
    synchronizes CPU/GPU threads, and introduces a brief settlement delay.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()  # Block CPU until GPU cache release finishes
    if delay_sec > 0:
        time.sleep(delay_sec)


# =============================================================================
# CONFIGURATION & METADATA SETUP
# =============================================================================

SYSTEM_PROMPT = "You are an expert telecommunications network engineer."
MODEL_NAME = getattr(general_model.config, "_name_or_path", "EssentialAI/rnj-1-instruct")
TIMESTAMP_STR = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = f"track1_general_llm_results_{TIMESTAMP_STR}.json"

# Ensure benchmark dataset exists
if "benchmark_questions" not in globals():
    raise NameError("Variable 'benchmark_questions' is missing. Load your benchmark dataset first.")

track1_general_results = []
track1_start = time.time()

# Initial hardware memory flush
flush_vram(delay_sec=1.0)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("BASELINE — TRACK 1 | GENERAL LLM ONLY (NO RAG)")
print("DIRECT INFERENCE")
print("=" * 90)
print(f"Model ID        : {MODEL_NAME}")
print(f"Questions       : {len(benchmark_questions)}")
print("=" * 90)


# =============================================================================
# EXECUTE ALL QUESTIONS
# =============================================================================

for index, item in enumerate(benchmark_questions, start=1):

    # 1. PRE-QUERY CLEANUP & SYNCHRONIZATION
    flush_vram(delay_sec=0.5)

    question_id = item["id"]
    category = item["category"]
    question = item["question"]

    print(f"\n[{index:02d}/{len(benchmark_questions):02d}] {question_id} | {category}")
    start_time = time.time()

    # =========================================================================
    # INFERENCE (wrapped in torch.inference_mode to disable autograd tracking)
    # =========================================================================
    try:
        with torch.inference_mode():
            # Generate standalone LLM response
            answer = generate_response(
                model=general_model,
                tokenizer=general_tokenizer,
                question=question,
                system_prompt=SYSTEM_PROMPT,
                max_new_tokens=GENERAL_GENERATION_CONFIG.get("max_new_tokens", 500),
                do_sample=GENERAL_GENERATION_CONFIG.get("do_sample", False),
                temperature=GENERAL_GENERATION_CONFIG.get("temperature", 0.01),
                repetition_penalty=GENERAL_GENERATION_CONFIG.get("repetition_penalty", 1.1),
            )

            # Compute input token length for telemetry tracking
            prompt_str = f"{SYSTEM_PROMPT}\n{question}"
            input_tokens = len(general_tokenizer.encode(prompt_str))
            output_tokens = len(general_tokenizer.encode(answer))

        elapsed_sec = time.time() - start_time

        record = {
            "question_id": question_id,
            "category": category,
            "question": question,
            "expected_points": item.get("expected_points", []),
            "model_id": MODEL_NAME,
            "mode": "LLM_ONLY",
            "status": "PASS",
            "answer": answer,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(elapsed_sec, 2),
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"  PASS | "
            f"Input: {input_tokens:,} | "
            f"Output: {output_tokens:,} | "
            f"Time: {elapsed_sec:.2f} sec"
        )

    except Exception as exc:
        elapsed_sec = time.time() - start_time

        # 2. ERROR RECOVERY CLEANUP
        flush_vram(delay_sec=0.5)

        record = {
            "question_id": question_id,
            "category": category,
            "question": question,
            "expected_points": item.get("expected_points", []),
            "model_id": MODEL_NAME,
            "mode": "LLM_ONLY",
            "status": "FAIL",
            "answer": None,
            "input_tokens": None,
            "output_tokens": None,
            "generation_time_sec": round(elapsed_sec, 2),
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": f"{type(exc).__name__}: {exc}",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(f"  FAIL | {type(exc).__name__}: {exc}")

    track1_general_results.append(record)

    # Incremental Checkpoint Save to disk
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(track1_general_results, f, indent=2, ensure_ascii=False)

    # 3. POST-QUERY CLEANUP
    flush_vram(delay_sec=0.2)


# =============================================================================
# SUMMARY & TELEMETRY
# =============================================================================

track1_elapsed = time.time() - track1_start
track1_pass = sum(r["status"] == "PASS" for r in track1_general_results)
track1_fail = sum(r["status"] == "FAIL" for r in track1_general_results)

# Create structured payload wrapper with top-level metadata
final_payload = {
    "metadata": {
        "track": "Track 1 - General LLM Only Baseline",
        "model_id": MODEL_NAME,
        "mode": "LLM_ONLY",
        "total_questions": len(benchmark_questions),
        "captured": len(track1_general_results),
        "pass_count": track1_pass,
        "fail_count": track1_fail,
        "total_runtime_minutes": round(track1_elapsed / 60, 2),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    },
    "results": track1_general_results
}

# Final metadata-wrapped save to disk
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_payload, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 90)
print("BASELINE — TRACK 1 BENCHMARK COMPLETE")
print("=" * 90)

print(f"Expected questions : {len(benchmark_questions)}")
print(f"Captured           : {len(track1_general_results)}")
print(f"PASS               : {track1_pass}")
print(f"FAIL               : {track1_fail}")
print(f"Runtime            : {track1_elapsed / 60:.2f} min")
print(f"Saved payload to   : {OUTPUT_FILE}")
print("=" * 90)

BASELINE — TRACK 1 | GENERAL LLM ONLY (NO RAG)
DIRECT INFERENCE
Model ID        : EssentialAI/rnj-1-instruct
Questions       : 20

[01/20] Q01 | 5G Core
  PASS | Input: 27 | Output: 461 | Time: 38.69 sec

[02/20] Q02 | 5G Core
  PASS | Input: 30 | Output: 886 | Time: 74.52 sec

[03/20] Q03 | 5G RAN
  PASS | Input: 29 | Output: 600 | Time: 50.55 sec

[04/20] Q04 | 5G RAN
  PASS | Input: 28 | Output: 467 | Time: 39.40 sec

[05/20] Q05 | 5G SA Procedures
  PASS | Input: 27 | Output: 979 | Time: 82.43 sec

[06/20] Q06 | 5G SA Procedures
  PASS | Input: 28 | Output: 520 | Time: 43.66 sec

[07/20] Q07 | Open RAN
  PASS | Input: 33 | Output: 774 | Time: 64.97 sec

[08/20] Q08 | Open RAN
  PASS | Input: 40 | Output: 773 | Time: 64.74 sec

[09/20] Q09 | Cloud-Native Telecom
  PASS | Input: 29 | Output: 611 | Time: 51.14 sec

[10/20] Q10 | Cloud-Native Telecom
  PASS | Input: 23 | Output: 336 | Time: 28.19 sec

[11/20] Q11 | Applied Telecom Engineering
  PASS | Input: 90 | Output: 1,025 | Time: 

### **Track 2 Inference**

In [14]:
import gc
import json
import time
from datetime import datetime, timezone
import torch

# =============================================================================
# HELPER: AGGRESSIVE HARDWARE SYNCHRONIZED MEMORY CLEARING
# =============================================================================

def flush_vram(delay_sec: float = 0.5):
    """
    Forces garbage collection, releases cached CUDA VRAM back to the GPU,
    synchronizes CPU/GPU threads, and introduces a brief settlement delay.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()  # Block CPU until GPU cache release finishes
    if delay_sec > 0:
        time.sleep(delay_sec)


# =============================================================================
# CONFIGURATION & METADATA SETUP
# =============================================================================

SYSTEM_PROMPT = "You are an expert telecommunications network engineer."
MODEL_NAME = getattr(general_model.config, "_name_or_path", "EssentialAI/rnj-1-instruct")
TIMESTAMP_STR = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = f"track2_general_llm_results_{TIMESTAMP_STR}.json"

# Ensure dataset exists
if "benchmark_questions_track2" not in globals():
    raise NameError("Variable 'benchmark_questions_track2' is missing. Load your Track 2 dataset first.")

track2_general_results = []
track2_start = time.time()

# Initial hardware memory flush
flush_vram(delay_sec=1.0)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("BASELINE — TRACK 2 | GENERAL LLM ONLY (NO RAG)")
print("DIRECT INFERENCE")
print("=" * 90)
print(f"Model ID        : {MODEL_NAME}")
print(f"Questions       : {len(benchmark_questions_track2)}")
print("=" * 90)


# =============================================================================
# EXECUTE BENCHMARK
# =============================================================================

for index, item in enumerate(benchmark_questions_track2, start=1):

    # 1. PRE-QUERY CLEANUP & SYNCHRONIZATION
    flush_vram(delay_sec=0.5)

    question_id = item["evaluation_id"]
    benchmark = item["benchmark"]
    question = item["question"]
    choices = item.get("choices")

    # Format model query without target answer leakage
    if choices:
        choices_text = "\n".join(str(choice) for choice in choices)
        model_query = f"{question}\n\nChoices:\n{choices_text}"
    else:
        model_query = question

    print(f"\n[{index:02d}/{len(benchmark_questions_track2):02d}] {question_id} | {benchmark}")
    start_time = time.time()

    # =========================================================================
    # INFERENCE (wrapped in torch.inference_mode to disable autograd tracking)
    # =========================================================================
    try:
        with torch.inference_mode():
            # Generate standalone LLM response
            answer = generate_response(
                model=general_model,
                tokenizer=general_tokenizer,
                question=model_query,
                system_prompt=SYSTEM_PROMPT,
                max_new_tokens=GENERAL_GENERATION_CONFIG.get("max_new_tokens", 500),
                do_sample=GENERAL_GENERATION_CONFIG.get("do_sample", False),
                temperature=GENERAL_GENERATION_CONFIG.get("temperature", 0.01),
                repetition_penalty=GENERAL_GENERATION_CONFIG.get("repetition_penalty", 1.1),
            )

            # Compute token telemetry
            prompt_str = f"{SYSTEM_PROMPT}\n{model_query}"
            input_tokens = len(general_tokenizer.encode(prompt_str))
            output_tokens = len(general_tokenizer.encode(answer))

        elapsed_sec = time.time() - start_time

        record = {
            "question_id": question_id,
            "benchmark": benchmark,
            "question": question,
            "choices": choices,
            "expected_answer": item.get("answer"),
            "explanation": item.get("explanation"),
            "candidate_selection_score": item.get("candidate_selection_score"),
            "model_id": MODEL_NAME,
            "mode": "LLM_ONLY",
            "status": "PASS",
            "answer": answer,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(elapsed_sec, 2),
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"  PASS | "
            f"Input: {input_tokens:,} | "
            f"Output: {output_tokens:,} | "
            f"Time: {elapsed_sec:.2f} sec"
        )

    except Exception as exc:
        elapsed_sec = time.time() - start_time

        # 2. ERROR RECOVERY CLEANUP
        flush_vram(delay_sec=0.5)

        record = {
            "question_id": question_id,
            "benchmark": benchmark,
            "question": question,
            "choices": choices,
            "expected_answer": item.get("answer"),
            "explanation": item.get("explanation"),
            "candidate_selection_score": item.get("candidate_selection_score"),
            "model_id": MODEL_NAME,
            "mode": "LLM_ONLY",
            "status": "FAIL",
            "answer": None,
            "input_tokens": None,
            "output_tokens": None,
            "generation_time_sec": round(elapsed_sec, 2),
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": f"{type(exc).__name__}: {exc}",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(f"  FAIL | {type(exc).__name__}: {exc}")

    track2_general_results.append(record)

    # Incremental Checkpoint Save to disk
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(track2_general_results, f, indent=2, ensure_ascii=False)

    # 3. POST-QUERY CLEANUP
    flush_vram(delay_sec=0.2)


# =============================================================================
# SUMMARY & TELEMETRY
# =============================================================================

track2_elapsed = time.time() - track2_start
track2_pass = sum(r["status"] == "PASS" for r in track2_general_results)
track2_fail = sum(r["status"] == "FAIL" for r in track2_general_results)

# Create structured payload wrapper with top-level metadata
final_payload = {
    "metadata": {
        "track": "Track 2 - General LLM Only Baseline",
        "model_id": MODEL_NAME,
        "mode": "LLM_ONLY",
        "total_questions": len(benchmark_questions_track2),
        "captured": len(track2_general_results),
        "pass_count": track2_pass,
        "fail_count": track2_fail,
        "total_runtime_minutes": round(track2_elapsed / 60, 2),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    },
    "results": track2_general_results
}

# Final metadata-wrapped save to disk
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_payload, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 90)
print("BASELINE — TRACK 2 BENCHMARK COMPLETE")
print("=" * 90)

print(f"Expected questions : {len(benchmark_questions_track2)}")
print(f"Captured           : {len(track2_general_results)}")
print(f"PASS               : {track2_pass}")
print(f"FAIL               : {track2_fail}")
print(f"Runtime            : {track2_elapsed / 60:.2f} min")
print(f"Artifacts saved    : {OUTPUT_FILE}")
print("=" * 90)

BASELINE — TRACK 2 | GENERAL LLM ONLY (NO RAG)
DIRECT INFERENCE
Model ID        : EssentialAI/rnj-1-instruct
Questions       : 32

[01/32] T2-01 | 3gpp_tsg
  PASS | Input: 1,289 | Output: 11 | Time: 1.90 sec

[02/32] T2-02 | 3gpp_tsg
  PASS | Input: 1,397 | Output: 11 | Time: 2.00 sec

[03/32] T2-03 | 3gpp_tsg
  PASS | Input: 1,284 | Output: 10 | Time: 1.81 sec

[04/32] T2-04 | 3gpp_tsg
  PASS | Input: 1,283 | Output: 10 | Time: 1.81 sec

[05/32] T2-05 | oranbench
  PASS | Input: 88 | Output: 498 | Time: 41.81 sec

[06/32] T2-06 | oranbench
  PASS | Input: 119 | Output: 425 | Time: 35.61 sec

[07/32] T2-07 | oranbench
  PASS | Input: 69 | Output: 184 | Time: 15.50 sec

[08/32] T2-08 | oranbench
  PASS | Input: 75 | Output: 245 | Time: 20.63 sec

[09/32] T2-09 | sixg_bench
  PASS | Input: 466 | Output: 521 | Time: 43.74 sec

[10/32] T2-10 | sixg_bench
  PASS | Input: 522 | Output: 1,025 | Time: 86.02 sec

[11/32] T2-11 | sixg_bench
  PASS | Input: 472 | Output: 1,025 | Time: 86.26 sec

